In [723]:
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

### Load and Structure

In [724]:
with open("results/sampling_benchmark_results.pkl", "rb") as f:
    sampling_results = pickle.load(f)
    
with open("results/no_benchmark_results.pkl", "rb") as f:
    no_results = pickle.load(f)
    
with open("results/hidden_benchmark_results.pkl", "rb") as f:
    hidden_results = pickle.load(f)
    
with open("results/observer_benchmark_results.pkl", "rb") as f:
    observer_results = pickle.load(f)

In [725]:
def build_domain_metric_tables(results_dict):
    """
    Parameters
    ----------
    results_dict : dict
        {
            "ModelName": {
                "model_name": ...,
                "y_true": ...,
                "y_pred": ...,
                "domain_id": ...,
                "fold_id": ...
            },
            ...
        }

    Returns
    -------
    metric_tables : dict of DataFrames
        {
            "accuracy": df,
            "precision": df,
            "recall": df,
            "f1": df
        }

        Each DataFrame: rows = models, columns = domains
    """

    # Collect all domains
    all_domains = set()
    for model_results in results_dict.values():
        all_domains.update(np.unique(model_results["domain_id"]))
    all_domains = sorted(all_domains)

    # Storage
    metrics_storage = {
        "accuracy": {},
        "precision": {},
        "recall": {},
        "f1": {}
    }

    for model_name, res in results_dict.items():

        y_true = np.array(res["y_true"])
        y_pred = np.array(res["y_pred"])
        domains = np.array(res["domain_id"])
        folds = np.array(res["fold_id"])

        for metric_name in metrics_storage.keys():
            metrics_storage[metric_name][model_name] = {}

        for domain in all_domains:

            domain_mask = domains == domain
            if np.sum(domain_mask) == 0:
                continue

            # Compute per-fold metric (correct aggregation)
            fold_scores = {
                "accuracy": [],
                "precision": [],
                "recall": [],
                "f1": []
            }

            for fold in np.unique(folds[domain_mask]):

                fold_mask = domain_mask & (folds == fold)

                yt = y_true[fold_mask]
                yp = y_pred[fold_mask]

                fold_scores["accuracy"].append(accuracy_score(yt, yp))
                fold_scores["precision"].append(
                    precision_score(yt, yp, zero_division=0)
                )
                fold_scores["recall"].append(
                    recall_score(yt, yp, zero_division=0)
                )
                fold_scores["f1"].append(
                    f1_score(yt, yp, zero_division=0)
                )

            # Store mean over folds
            for metric_name in metrics_storage.keys():
                metrics_storage[metric_name][model_name][domain] = np.mean(
                    fold_scores[metric_name]
                )

    # Convert to DataFrames
    metric_tables = {}
    for metric_name, data in metrics_storage.items():
        df = pd.DataFrame(data).T
        df = df[all_domains]  # consistent column order
        metric_tables[metric_name] = df

    return metric_tables


In [726]:
def add_mean_std_columns(df):
    """
    Adds 'mean' and 'std' columns computed across domain columns.

    Assumes:
    - Rows = methods
    - Columns = domains
    """

    df_with_stats = df.copy()

    # Row-wise statistics (across domains)
    df_with_stats["mean"] = df_with_stats.mean(axis=1)
    df_with_stats["std"] = df_with_stats.std(axis=1)

    return df_with_stats


### No shift

In [727]:
tables_no = build_domain_metric_tables(no_results)
add_mean_std_columns(tables_no["f1"])

,0.0,mean,std
Logistic,0.996004,0.996004,0.0
RandomForest,0.872655,0.872655,0.0
SVM,0.965000,0.965000,0.0
NN_ERM,0.967484,0.967484,0.0
NN_L2,0.970090,0.970090,0.0
NN_Dropout,0.985075,0.985075,0.0


In [728]:
add_mean_std_columns(tables_no["accuracy"])

,0.0,mean,std
Logistic,0.9960,0.9960,0.0
RandomForest,0.8710,0.8710,0.0
SVM,0.9650,0.9650,0.0
NN_ERM,0.9675,0.9675,0.0
NN_L2,0.9700,0.9700,0.0
NN_Dropout,0.9850,0.9850,0.0


### Sampling shift

In [729]:
tables_sampling = build_domain_metric_tables(sampling_results)
add_mean_std_columns(tables_sampling["f1"])

,0,1,2,3,4,5,6,7,8,9,mean,std
Logistic,0.971429,0.995696,0.998895,0.969940,0.996516,0.981818,0.990291,0.998935,0.934911,0.990741,0.982917,0.018926
RandomForest,0.443396,0.845745,0.864734,0.695906,0.915777,0.644068,0.257143,0.974843,0.476190,0.609195,0.672700,0.220561
SVM,0.867133,0.975110,0.981502,0.908046,0.978723,0.871795,0.886792,0.986359,0.870056,0.926829,0.925235,0.048273
NN_ERM,0.819277,0.960236,0.982456,0.906250,0.981221,0.924188,0.875000,0.993603,0.891720,0.884319,0.921827,0.053946
NN_L2,0.757062,0.950673,0.986726,0.916828,0.957780,0.920415,0.892857,0.992545,0.872928,0.922353,0.917017,0.064429
NN_Dropout,0.850000,0.984308,0.992333,0.926923,0.976359,0.940767,0.943396,0.996805,0.916667,0.942029,0.946959,0.041950
VREx,0.867133,0.975680,0.987791,0.922449,0.941176,0.935252,0.910714,0.994687,0.838710,0.925301,0.929889,0.047435
DANN,0.810811,0.963687,0.981421,0.908722,0.970273,0.946619,0.875000,0.995726,0.821622,0.911447,0.918533,0.062013
SVM_DomainMajority,0.626728,0.968100,0.974138,0.841739,0.988453,0.774929,0.838710,0.992608,0.790000,0.957111,0.875251,0.115431
NN_SVM_Global,0.816901,0.928025,0.978865,0.929134,0.966587,0.833333,0.867925,0.994687,0.865169,0.906977,0.908760,0.058418


In [730]:
add_mean_std_columns(tables_sampling["accuracy"])

,0,1,2,3,4,5,6,7,8,9,mean,std
Logistic,0.992,0.994,0.998,0.970,0.994,0.990,0.998,0.998,0.978,0.992,0.9904,0.008800
RandomForest,0.764,0.768,0.776,0.584,0.858,0.790,0.896,0.952,0.714,0.728,0.7830,0.097481
SVM,0.962,0.966,0.966,0.904,0.964,0.920,0.976,0.974,0.954,0.940,0.9526,0.022734
NN_ERM,0.940,0.946,0.968,0.916,0.968,0.958,0.972,0.988,0.966,0.910,0.9532,0.023803
NN_L2,0.914,0.934,0.976,0.914,0.930,0.954,0.976,0.986,0.954,0.934,0.9472,0.024710
NN_Dropout,0.952,0.978,0.986,0.924,0.960,0.966,0.988,0.994,0.972,0.952,0.9672,0.020064
VREx,0.962,0.966,0.978,0.924,0.904,0.964,0.980,0.990,0.940,0.938,0.9546,0.025893
DANN,0.944,0.948,0.966,0.910,0.950,0.970,0.972,0.992,0.934,0.918,0.9504,0.024130
SVM_DomainMajority,0.838,0.954,0.952,0.818,0.980,0.842,0.960,0.986,0.916,0.962,0.9208,0.060598
NN_SVM_Global,0.948,0.906,0.962,0.928,0.944,0.892,0.972,0.990,0.952,0.920,0.9414,0.028664


### Hidden shift

In [731]:
tables_hidden = build_domain_metric_tables(hidden_results)
add_mean_std_columns(tables_hidden["f1"])

,0,1,2,3,4,5,6,7,8,9,mean,std
Logistic,0.843462,0.724444,0.803987,0.833333,0.634643,0.681063,0.544444,0.944056,0.698020,0.403361,0.711081,0.149977
RandomForest,0.749035,0.699571,0.752577,0.721992,0.538941,0.565766,0.510638,0.794776,0.629464,0.355556,0.631832,0.130996
SVM,0.811321,0.726862,0.770017,0.778723,0.634643,0.658784,0.552113,0.884135,0.681159,0.401114,0.689887,0.132146
NN_ERM,0.750943,0.700229,0.738095,0.717622,0.656160,0.668896,0.544413,0.777778,0.653846,0.423529,0.663151,0.101238
NN_L2,0.745698,0.686916,0.745299,0.718615,0.646465,0.685338,0.565476,0.784530,0.658768,0.400000,0.663710,0.105599
NN_Dropout,0.749004,0.762353,0.714286,0.808126,0.626647,0.638079,0.576471,0.848030,0.703242,0.405634,0.683187,0.121822
VREx,0.764259,0.728132,0.776119,0.708155,0.675141,0.692683,0.531792,0.779599,0.666667,0.393443,0.671599,0.115371
DANN,0.758491,0.675862,0.755034,0.692946,0.677010,0.694669,0.555556,0.787986,0.652381,0.407932,0.665787,0.105882
SVM_DomainMajority,0.782946,0.739229,0.752166,0.836364,0.614476,0.645051,0.549020,0.903461,0.712121,0.411429,0.694626,0.136891
NN_SVM_Global,0.753247,0.666667,0.710345,0.670886,0.669504,0.650000,0.518732,0.750916,0.596330,0.395349,0.638197,0.104473


In [732]:
add_mean_std_columns(tables_hidden["accuracy"])

,0,1,2,3,4,5,6,7,8,9,mean,std
Logistic,0.830,0.752,0.764,0.852,0.498,0.616,0.672,0.936,0.756,0.574,0.7250,0.128026
RandomForest,0.740,0.720,0.712,0.732,0.408,0.518,0.632,0.780,0.668,0.478,0.6388,0.120596
SVM,0.800,0.758,0.730,0.792,0.498,0.596,0.682,0.870,0.736,0.570,0.7032,0.110302
NN_ERM,0.736,0.738,0.692,0.734,0.520,0.604,0.682,0.760,0.712,0.608,0.6786,0.073163
NN_L2,0.734,0.732,0.702,0.740,0.510,0.618,0.708,0.766,0.712,0.568,0.6790,0.080116
NN_Dropout,0.748,0.798,0.680,0.830,0.490,0.578,0.712,0.838,0.762,0.578,0.7014,0.112289
VREx,0.752,0.770,0.730,0.728,0.540,0.622,0.676,0.758,0.724,0.556,0.6856,0.080094
DANN,0.744,0.718,0.708,0.704,0.542,0.622,0.696,0.760,0.708,0.582,0.6784,0.068063
SVM_DomainMajority,0.776,0.770,0.714,0.856,0.478,0.584,0.678,0.894,0.772,0.588,0.7110,0.123526
NN_SVM_Global,0.734,0.708,0.664,0.688,0.534,0.580,0.666,0.728,0.648,0.584,0.6534,0.064032


### Observer shift

In [733]:
tables_observer = build_domain_metric_tables(observer_results)
add_mean_std_columns(tables_observer["f1"])

,0,1,2,3,4,5,6,7,8,9,mean,std
Logistic,0.852814,0.873786,0.837398,0.837782,0.827586,0.851927,0.858144,0.838323,0.839757,0.835294,0.845281,0.012996
RandomForest,0.765591,0.818533,0.718367,0.774704,0.747433,0.807767,0.758748,0.777555,0.813360,0.824663,0.780672,0.033032
SVM,0.844639,0.847059,0.800000,0.847390,0.828974,0.846457,0.823117,0.823770,0.826884,0.827853,0.831614,0.014332
NN_ERM,0.808989,0.832998,0.792079,0.792608,0.777328,0.837302,0.811287,0.823045,0.775681,0.798403,0.804972,0.020488
NN_L2,0.816594,0.842315,0.790514,0.804124,0.783133,0.836735,0.817204,0.812000,0.784232,0.811765,0.809862,0.019135
NN_Dropout,0.836207,0.871094,0.809524,0.831967,0.810277,0.846774,0.850088,0.826176,0.826446,0.833659,0.834221,0.017569
VREx,0.800885,0.828000,0.796000,0.818363,0.764228,0.826176,0.823105,0.827586,0.802444,0.797595,0.808438,0.019229
DANN,0.802661,0.836292,0.785425,0.796844,0.783133,0.821138,0.817052,0.805726,0.770526,0.811133,0.802993,0.018720
SVM_DomainMajority,0.820399,0.866019,0.791753,0.832347,0.790984,0.848126,0.797101,0.800000,0.823762,0.845124,0.821562,0.024993
NN_SVM_Global,0.788793,0.812749,0.786561,0.770791,0.771429,0.808081,0.811092,0.814815,0.773663,0.783626,0.792160,0.017009


In [734]:
add_mean_std_columns(tables_observer["accuracy"])

,0,1,2,3,4,5,6,7,8,9,mean,std
Logistic,0.864,0.870,0.840,0.842,0.830,0.854,0.838,0.838,0.842,0.832,0.8450,0.012657
RandomForest,0.782,0.812,0.724,0.772,0.754,0.802,0.738,0.778,0.810,0.818,0.7790,0.030806
SVM,0.858,0.844,0.804,0.848,0.830,0.844,0.798,0.828,0.830,0.822,0.8306,0.018068
NN_ERM,0.830,0.834,0.790,0.798,0.780,0.836,0.786,0.828,0.786,0.798,0.8066,0.021449
NN_L2,0.832,0.842,0.788,0.810,0.784,0.840,0.796,0.812,0.792,0.808,0.8104,0.020235
NN_Dropout,0.848,0.868,0.808,0.836,0.808,0.848,0.830,0.830,0.832,0.830,0.8338,0.017192
VREx,0.820,0.828,0.796,0.818,0.768,0.830,0.804,0.830,0.806,0.798,0.8098,0.018557
DANN,0.822,0.834,0.788,0.794,0.784,0.824,0.794,0.810,0.782,0.810,0.8042,0.017423
SVM_DomainMajority,0.838,0.862,0.798,0.830,0.796,0.846,0.776,0.802,0.822,0.838,0.8208,0.025506
NN_SVM_Global,0.804,0.812,0.784,0.774,0.776,0.810,0.782,0.820,0.780,0.778,0.7920,0.016541
